Predicting Feature similarity and adaption weights

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

df = pd.read_csv('userknowledge.csv', header=None)

feature_cols = ['F1', 'F2', 'F3', 'F4', 'F5']
df.columns = feature_cols + ['Class']
num_features = len(feature_cols)

X_raw = df[feature_cols].values

le = LabelEncoder()
y_raw = le.fit_transform(df['Class'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

folds = []
for train_idx, test_idx in skf.split(X_scaled, y_raw):
    X_tr, y_tr = X_scaled[train_idx], y_raw[train_idx]
    X_te, y_te = X_scaled[test_idx], y_raw[test_idx]
    
    diff_tr = X_tr[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    diff_te = X_te[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    
    folds.append({
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_te': X_te, 'y_te': y_te,
        'diff_tr_sq': diff_tr ** 2,
        'diff_te_sq': diff_te ** 2
    })

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_sim(trial):
    weights = np.array([trial.suggest_float(f'w_{col}', 0.0, 1.0) for col in feature_cols])
    
    w_sum = np.sum(weights)
    if w_sum == 0: return 0.0
    weights = weights / w_sum

    mean_train_acc = 0
    for fold in folds:
        dist = np.sum(weights * fold['diff_tr_sq'], axis=2)
        np.fill_diagonal(dist, np.inf)
        
        nn_indices = np.argmin(dist, axis=1)
        preds = fold['y_tr'][nn_indices]
        mean_train_acc += accuracy_score(fold['y_tr'], preds)
        
    return mean_train_acc / 10

print("Starting Optuna optimization for Similarity Weights (10-fold Avg LOOCV)")
study_sim = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_sim.optimize(objective_sim, n_trials=10000)

best_w_raw = np.array([study_sim.best_params[f'w_{col}'] for col in feature_cols])
best_w = best_w_raw / np.sum(best_w_raw)
print(f"Optimization Complete. Best 10-Fold Avg LOOCV Accuracy: {study_sim.best_value * 100:.2f}%\n")

print("Optimized Similarity Weights (w):")
for name, weight in zip(feature_cols, best_w):
    print(f"  {name:15s}: {weight:.4f}")
print("\n")

metrics = {
    'train_uni': 0, 'test_uni': 0,
    'train_opt_w': 0, 'test_opt_w': 0
}

uniform_w = np.ones(num_features) / num_features

print(f"{'Fold':<6} | {'Uniform Sim Acc (Train / Test)':<32} | {'Opt Sim Acc (Train / Test)':<30}")

for i, fold in enumerate(folds):
    dist_tr_uni = np.sum(uniform_w * fold['diff_tr_sq'], axis=2)
    np.fill_diagonal(dist_tr_uni, np.inf)
    nn_tr_uni = np.argmin(dist_tr_uni, axis=1)
    train_uni_f = accuracy_score(fold['y_tr'], fold['y_tr'][nn_tr_uni])
    
    dist_te_uni = np.sum(uniform_w * fold['diff_te_sq'], axis=2)
    nn_te_uni = np.argmin(dist_te_uni, axis=1)
    test_uni_f = accuracy_score(fold['y_te'], fold['y_tr'][nn_te_uni])

    dist_tr_opt = np.sum(best_w * fold['diff_tr_sq'], axis=2)
    np.fill_diagonal(dist_tr_opt, np.inf)
    nn_tr_opt = np.argmin(dist_tr_opt, axis=1)
    train_opt_f = accuracy_score(fold['y_tr'], fold['y_tr'][nn_tr_opt])
    
    dist_te_opt = np.sum(best_w * fold['diff_te_sq'], axis=2)
    nn_te_opt = np.argmin(dist_te_opt, axis=1)
    test_opt_f = accuracy_score(fold['y_te'], fold['y_tr'][nn_te_opt])
    
    print(f"Fold {i+1:<2} | {train_uni_f:>13.4f} / {test_uni_f:<14.4f} | {train_opt_f:>11.4f} / {test_opt_f:<12.4f}")

    metrics['train_uni'] += train_uni_f
    metrics['test_uni'] += test_uni_f
    metrics['train_opt_w'] += train_opt_f
    metrics['test_opt_w'] += test_opt_f

for k in metrics: 
    metrics[k] /= 10

print("\n10-FOLD AVERAGE RESULTS")
print(f"Uniform Similarity Baseline:")
print(f"  Avg Train LOOCV Accuracy : {metrics['train_uni'] * 100:.2f}%")
print(f"  Avg Test Accuracy        : {metrics['test_uni'] * 100:.2f}%\n")

print(f"Optimized Similarity Weights:")
print(f"  Avg Train LOOCV Accuracy : {metrics['train_opt_w'] * 100:.2f}%")
print(f"  Avg Test Accuracy        : {metrics['test_opt_w'] * 100:.2f}%\n")

if metrics['test_uni'] > 0:
    improvement = ((metrics['test_opt_w'] - metrics['test_uni']) / metrics['test_uni']) * 100
    print(f"Overall Unseen Test Accuracy Improvement (Uniform -> Optimized): {improvement:.2f}%")

/opt/miniconda3/envs/mt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Optuna optimization for Similarity Weights (10-fold Avg LOOCV)
Optimization Complete. Best 10-Fold Avg LOOCV Accuracy: 92.64%

Optimized Similarity Weights (w):
  F1             : 0.0003
  F2             : 0.0042
  F3             : 0.0001
  F4             : 0.4608
  F5             : 0.5346


Fold   | Uniform Sim Acc (Train / Test)   | Opt Sim Acc (Train / Test)    
Fold 1  |        0.7956 / 0.7805         |      0.9199 / 0.9268      
Fold 2  |        0.7762 / 0.7317         |      0.9254 / 0.9268      
Fold 3  |        0.7707 / 0.7317         |      0.9254 / 0.9512      
Fold 4  |        0.7989 / 0.7500         |      0.9449 / 0.8250      
Fold 5  |        0.7906 / 0.7500         |      0.9256 / 0.9000      
Fold 6  |        0.7824 / 0.8250         |      0.9311 / 0.9000      
Fold 7  |        0.7934 / 0.7500         |      0.9229 / 0.9250      
Fold 8  |        0.7769 / 0.8000         |      0.9366 / 0.9500      
Fold 9  |        0.7934 / 0.8000         |      0.9174 / 0.9750

Simulating subsets as the size of footprint set of that fold for all the 10 folds in both 80-20 and 50-50 fashion

In [ ]:
import numpy as np
import pandas as pd
import os
import networkx as nx
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

def get_train_footprint(X_tr, y_tr, sim_weights):
    n_cases = len(X_tr)
    
    diff_tr = X_tr[:, np.newaxis, :] - X_tr[np.newaxis, :, :]
    dist = np.sum(sim_weights * (diff_tr ** 2), axis=2)
    np.fill_diagonal(dist, np.inf) # Ignore self in LOOCV
    
    nn_idx = np.argmin(dist, axis=1)
    preds = y_tr[nn_idx]
    
    coverage = {i: set([i]) for i in range(n_cases)}
    reachability = {i: set([i]) for i in range(n_cases)}

    for i in range(n_cases):
        actual_class = y_tr[i]
        predicted_class = preds[i]
        best_nn_idx = nn_idx[i]
        
        if predicted_class == actual_class:
            coverage[best_nn_idx].add(i)
            reachability[i].add(best_nn_idx)

    rc = np.zeros(n_cases)
    for i in range(n_cases):
        for covered_idx in coverage[i]:
            if len(reachability[covered_idx]) > 0:
                rc[i] += 1.0 / len(reachability[covered_idx])

    related_sets = {i: coverage[i].union(reachability[i]) for i in range(n_cases)}
    G = nx.Graph()
    G.add_nodes_from(range(n_cases))
    
    for i in range(n_cases):
        for j in range(i + 1, n_cases):
            if len(related_sets[i].intersection(related_sets[j])) > 0:
                G.add_edge(i, j)

    global_footprint = set()
    for group in nx.connected_components(G):
        sorted_group = sorted(list(group), key=lambda x: (rc[x], len(coverage[x]), x), reverse=True)
        group_fp = set()
        changes = True
        while changes:
            changes = False
            for c_idx in sorted_group[:]:
                is_covered = any(c_idx in coverage[fp_idx] for fp_idx in group_fp)
                if not is_covered:
                    group_fp.add(c_idx)
                    sorted_group.remove(c_idx)
                    changes = True
        global_footprint.update(group_fp)

    return global_footprint

if __name__ == "__main__":
    
    best_w = np.array([0.0003, 0.0042, 0.0001, 0.4608, 0.5346], dtype=float) 

    df = pd.read_csv('userknowledge.csv', header=None)
    feature_cols = ['F1', 'F2', 'F3', 'F4', 'F5']
    df.columns = feature_cols + ['Class']
    
    X_raw = df[feature_cols].values
    
    le = LabelEncoder()
    y_raw = le.fit_transform(df['Class'])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    save_dir = 'saved_subset_userknowledge_algorithm_1'
    os.makedirs(save_dir, exist_ok=True)
    np.random.seed(42)

    for fold_id, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_raw)):
        print(f"\nPROCESSING FOLD {fold_id + 1}")
        
        X_tr, y_tr = X_scaled[train_idx], y_raw[train_idx]
        n_train = len(X_tr)

        fp_indices = list(get_train_footprint(X_tr, y_tr, best_w))
        fp_size = len(fp_indices)
        print(f"Footprint Size: {fp_size} cases (out of {n_train} training cases)\n")
        
        fp_cases_concat = np.hstack((X_tr[fp_indices], y_tr[fp_indices].reshape(-1, 1)))
        np.save(f'{save_dir}/original_footprint_fold_{fold_id}.npy', fp_cases_concat)

        prob_scenario_1 = np.zeros(n_train)
        prob_scenario_1[fp_indices] = 0.8
        non_fp_indices = list(set(range(n_train)) - set(fp_indices))
        if non_fp_indices:
            prob_scenario_1[non_fp_indices] = 0.2
        prob_scenario_1 /= prob_scenario_1.sum()

        print("SCENARIO 1: Biased Selection (0.2 / 0.8)")
        for iteration in range(1, 11):
            subset_idx = np.random.choice(n_train, size=fp_size, replace=False, p=prob_scenario_1)
            subset_cases_concat = np.hstack((X_tr[subset_idx], y_tr[subset_idx].reshape(-1, 1)))
            np.save(f'{save_dir}/original_footprint_fold_{fold_id}_scenario_1_iter_{iteration}.npy', subset_cases_concat)
            
            missing_cases = len(set(fp_indices) - set(subset_idx))
            print(f"  Iteration {iteration:2d} | Missing Original FP Cases: {missing_cases}")

        prob_scenario_2 = np.ones(n_train) * 0.5
        prob_scenario_2 /= prob_scenario_2.sum()

        print("\nSCENARIO 2: Uniform Random Selection (0.5)")
        for iteration in range(1, 11):
            subset_idx = np.random.choice(n_train, size=fp_size, replace=False, p=prob_scenario_2)
            subset_cases_concat = np.hstack((X_tr[subset_idx], y_tr[subset_idx].reshape(-1, 1)))
            np.save(f'{save_dir}/original_footprint_fold_{fold_id}_scenario_2_iter_{iteration}.npy', subset_cases_concat)
            
            missing_cases = len(set(fp_indices) - set(subset_idx))
            print(f"  Iteration {iteration:2d} | Missing Original FP Cases: {missing_cases}")


PROCESSING FOLD 1
Footprint Size: 170 cases (out of 362 training cases)

SCENARIO 1: Biased Selection (0.2 / 0.8)
  Iteration  1 | Missing Original FP Cases: 38
  Iteration  2 | Missing Original FP Cases: 55
  Iteration  3 | Missing Original FP Cases: 43
  Iteration  4 | Missing Original FP Cases: 51
  Iteration  5 | Missing Original FP Cases: 54
  Iteration  6 | Missing Original FP Cases: 48
  Iteration  7 | Missing Original FP Cases: 48
  Iteration  8 | Missing Original FP Cases: 50
  Iteration  9 | Missing Original FP Cases: 53
  Iteration 10 | Missing Original FP Cases: 48

SCENARIO 2: Uniform Random Selection (0.5)
  Iteration  1 | Missing Original FP Cases: 87
  Iteration  2 | Missing Original FP Cases: 88
  Iteration  3 | Missing Original FP Cases: 83
  Iteration  4 | Missing Original FP Cases: 93
  Iteration  5 | Missing Original FP Cases: 88
  Iteration  6 | Missing Original FP Cases: 90
  Iteration  7 | Missing Original FP Cases: 91
  Iteration  8 | Missing Original FP Cases

Evaluation and statistics after adding the missing cases of the footprint sets (Between original subset and added subset)

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from scipy.stats import ttest_rel

def evaluate_subset(X_subset, y_subset, X_te, y_te, sim_weights):
    diff = X_te[:, np.newaxis, :] - X_subset[np.newaxis, :, :]
    dist = np.sum(sim_weights * (diff ** 2), axis=2)
    nn_idx = np.argmin(dist, axis=1)
    
    preds = y_subset[nn_idx]
    
    return accuracy_score(y_te, preds)

def get_missing_cases(fp_cases, subset_cases):
    missing = []
    for fp_row in fp_cases:
        if not np.any(np.all(np.isclose(subset_cases, fp_row, atol=1e-5), axis=1)):
            missing.append(fp_row)
    return np.array(missing) if len(missing) > 0 else np.empty((0, fp_cases.shape[1]))

if __name__ == "__main__":
    
    best_w = np.array([0.0003, 0.0042, 0.0001, 0.4608, 0.5346], dtype=float)

    df = pd.read_csv('userknowledge.csv', header=None)
    feature_cols = ['F1', 'F2', 'F3', 'F4', 'F5']
    df.columns = feature_cols + ['Class']
    
    X_raw = df[feature_cols].values
    
    le = LabelEncoder()
    y_raw = le.fit_transform(df['Class'])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    save_dir = 'saved_subset_userknowledge_algorithm_1'
    
    global_better = 0
    global_same = 0
    global_worse = 0
    global_total = 0
    
    global_unadded_accs = []
    global_added_accs = []
    global_added_counts = []
    global_added_pcts = []
    
    global_scen1_unadded = []
    global_scen1_added = []
    global_scen1_added_counts = []
    global_scen1_added_pcts = []
    
    global_scen2_unadded = []
    global_scen2_added = []
    global_scen2_added_counts = []
    global_scen2_added_pcts = []

    global_scen1_better = 0
    global_scen1_same = 0
    global_scen1_worse = 0

    global_scen2_better = 0
    global_scen2_same = 0
    global_scen2_worse = 0

    for fold_id, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_raw)):
        print(f"\nEVALUATING FOLD {fold_id + 1}")
        
        X_tr_complete, y_tr_complete = X_scaled[train_idx], y_raw[train_idx]
        X_te, y_te = X_scaled[test_idx], y_raw[test_idx]

        complete_acc = evaluate_subset(X_tr_complete, y_tr_complete, X_te, y_te, best_w)
        print(f"Complete Train Data Test Accuracy: {complete_acc:.4f}")

        fp_path = f'{save_dir}/original_footprint_fold_{fold_id}.npy'
        if not os.path.exists(fp_path):
            print(f"Missing {fp_path}, skipping fold...")
            continue
            
        fp_data = np.load(fp_path)
        print(f"Original Footprint Size: {len(fp_data)}")
        
        X_fp, y_fp = fp_data[:, :-1], fp_data[:, -1]
        fp_acc = evaluate_subset(X_fp, y_fp, X_te, y_te, best_w)
        print(f"Original Footprint Test Accuracy : {fp_acc:.4f}")
        
        fold_better = 0
        fold_same = 0
        fold_worse = 0
        
        fold_unadded_accs = []
        fold_added_accs = []
        fold_added_counts = []
        fold_added_pcts = []

        for scenario in [1, 2]:
            print(f"\nSCENARIO {scenario}")
            scenario_unadded_accs = []
            scenario_added_accs = []
            scenario_added_counts = []
            scenario_added_pcts = []
            
            for iteration in range(1, 11):
                subset_path = f'{save_dir}/original_footprint_fold_{fold_id}_scenario_{scenario}_iter_{iteration}.npy'
                
                if not os.path.exists(subset_path):
                    continue
                
                subset_data = np.load(subset_path)
                X_sub, y_sub = subset_data[:, :-1], subset_data[:, -1]
                
                unadded_acc = evaluate_subset(X_sub, y_sub, X_te, y_te, best_w)
    
                missing_cases = get_missing_cases(fp_data, subset_data)
                added_count = len(missing_cases)
                added_pct = (added_count / len(fp_data)) * 100.0 if len(fp_data) > 0 else 0.0
                
                if added_count > 0:
                    added_data = np.vstack((subset_data, missing_cases))
                    X_add, y_add = added_data[:, :-1], added_data[:, -1]
                    added_acc = evaluate_subset(X_add, y_add, X_te, y_te, best_w)
                else:
                    added_data = subset_data
                    added_acc = unadded_acc

                updated_path = f'{save_dir}/original_footprint_fold_{fold_id}_scenario_{scenario}_iter_{iteration}_updated.npy'
                np.save(updated_path, added_data)

                scenario_unadded_accs.append(unadded_acc)
                scenario_added_accs.append(added_acc)
                scenario_added_counts.append(added_count)
                scenario_added_pcts.append(added_pct)

                if np.isclose(added_acc, unadded_acc, atol=1e-8):
                    fold_same += 1
                    if scenario == 1: global_scen1_same += 1
                    elif scenario == 2: global_scen2_same += 1
                elif added_acc > unadded_acc:
                    fold_better += 1
                    if scenario == 1: global_scen1_better += 1
                    elif scenario == 2: global_scen2_better += 1
                else:
                    fold_worse += 1
                    if scenario == 1: global_scen1_worse += 1
                    elif scenario == 2: global_scen2_worse += 1
                
                print(f"Iter {iteration:2d} | Unadded Acc: {unadded_acc:6.4f} | "
                      f"Missing FP Added: {added_count:2d} ({added_pct:5.2f}%) -> New Size: {len(subset_data)+added_count:3d} | "
                      f"Added Acc: {added_acc:6.4f}")
            
            if len(scenario_unadded_accs) > 0:
                scen_avg_unadded = np.mean(scenario_unadded_accs)
                scen_avg_added = np.mean(scenario_added_accs)
                scen_avg_cases = np.mean(scenario_added_counts)
                scen_avg_pcts = np.mean(scenario_added_pcts)
                print(f"Scenario {scenario} Avg Unadded Acc: {scen_avg_unadded:.4f} | Avg Added Acc: {scen_avg_added:.4f}")
                print(f"Scenario {scenario} Avg Cases Added: {scen_avg_cases:.2f} ({scen_avg_pcts:.2f}%)")
                
                if len(scenario_unadded_accs) > 1:
                    t_stat, p_val = ttest_rel(scenario_unadded_accs, scenario_added_accs)
                    print(f"Scenario {scenario} Paired T-Test: t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")

                fold_unadded_accs.extend(scenario_unadded_accs)
                fold_added_accs.extend(scenario_added_accs)
                fold_added_counts.extend(scenario_added_counts)
                fold_added_pcts.extend(scenario_added_pcts)
                
                if scenario == 1:
                    global_scen1_unadded.extend(scenario_unadded_accs)
                    global_scen1_added.extend(scenario_added_accs)
                    global_scen1_added_counts.extend(scenario_added_counts)
                    global_scen1_added_pcts.extend(scenario_added_pcts)
                elif scenario == 2:
                    global_scen2_unadded.extend(scenario_unadded_accs)
                    global_scen2_added.extend(scenario_added_accs)
                    global_scen2_added_counts.extend(scenario_added_counts)
                    global_scen2_added_pcts.extend(scenario_added_pcts)

        print(f"\nFOLD {fold_id + 1} SUMMARY:")
        if len(fold_unadded_accs) > 0:
            fold_avg_unadded = np.mean(fold_unadded_accs)
            fold_avg_added = np.mean(fold_added_accs)
            fold_avg_cases = np.mean(fold_added_counts)
            fold_avg_pcts = np.mean(fold_added_pcts)
            print(f"Fold Avg Unadded Acc: {fold_avg_unadded:.4f} | Fold Avg Added Acc: {fold_avg_added:.4f}")
            print(f"Fold Avg Cases Added: {fold_avg_cases:.2f} ({fold_avg_pcts:.2f}%)")
            
            if len(fold_unadded_accs) > 1:
                t_stat, p_val = ttest_rel(fold_unadded_accs, fold_added_accs)
                print(f"Fold Paired T-Test: t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")

            global_unadded_accs.extend(fold_unadded_accs)
            global_added_accs.extend(fold_added_accs)
            global_added_counts.extend(fold_added_counts)
            global_added_pcts.extend(fold_added_pcts)
            
        print(f"Better: {fold_better} | Same: {fold_same} | Worse: {fold_worse} (Out of {fold_better + fold_same + fold_worse})")
        
        global_better += fold_better
        global_same += fold_same
        global_worse += fold_worse
        global_total += (fold_better + fold_same + fold_worse)

    print(f"\nGLOBAL SUMMARY")

    print(f"\n[GLOBAL SCENARIO 1]")
    if len(global_scen1_unadded) > 0:
        print(f"Avg Unadded Acc             : {np.mean(global_scen1_unadded):.4f}")
        print(f"Avg Added Acc               : {np.mean(global_scen1_added):.4f}")
        print(f"Avg Cases Added             : {np.mean(global_scen1_added_counts):.2f} ({np.mean(global_scen1_added_pcts):.2f}%)")
        print(f"Accuracy Increased (Improved): {global_scen1_better}")
        print(f"Accuracy Remained the Same  : {global_scen1_same}")
        print(f"Accuracy Decreased (Worsened): {global_scen1_worse}")
        if len(global_scen1_unadded) > 1:
            t_stat, p_val = ttest_rel(global_scen1_unadded, global_scen1_added)
            print(f"Paired T-Test               : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
    else:
        print("No data for Scenario 1.")

    print(f"\n[GLOBAL SCENARIO 2]")
    if len(global_scen2_unadded) > 0:
        print(f"Avg Unadded Acc             : {np.mean(global_scen2_unadded):.4f}")
        print(f"Avg Added Acc               : {np.mean(global_scen2_added):.4f}")
        print(f"Avg Cases Added             : {np.mean(global_scen2_added_counts):.2f} ({np.mean(global_scen2_added_pcts):.2f}%)")
        print(f"Accuracy Increased (Improved): {global_scen2_better}")
        print(f"Accuracy Remained the Same  : {global_scen2_same}")
        print(f"Accuracy Decreased (Worsened): {global_scen2_worse}")
        if len(global_scen2_unadded) > 1:
            t_stat, p_val = ttest_rel(global_scen2_unadded, global_scen2_added)
            print(f"Paired T-Test               : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
    else:
        print("No data for Scenario 2.")

    print(f"\n[ENTIRE PROCESS (Overall)]")
    if len(global_unadded_accs) > 0:
        global_avg_unadded = np.mean(global_unadded_accs)
        global_avg_added = np.mean(global_added_accs)
        print(f"Avg Unadded Acc             : {global_avg_unadded:.4f}")
        print(f"Avg Added Acc               : {global_avg_added:.4f}")
        print(f"Avg Cases Added             : {np.mean(global_added_counts):.2f} ({np.mean(global_added_pcts):.2f}%)")
        
        if len(global_unadded_accs) > 1:
            t_stat, p_val = ttest_rel(global_unadded_accs, global_added_accs)
            print(f"Paired T-Test               : t-stat = {t_stat:.4f}, p-value = {p_val:.4e}")
            
    print(f"Accuracy Increased (Improved): {global_better}")
    print(f"Accuracy Remained the Same  : {global_same}")
    print(f"Accuracy Decreased (Worsened): {global_worse}")
    print(f"Total Evaluated Subsets     : {global_total}")


EVALUATING FOLD 1
Complete Train Data Test Accuracy: 0.9268
Original Footprint Size: 170
Original Footprint Test Accuracy : 0.9024

SCENARIO 1
Iter  1 | Unadded Acc: 0.9024 | Missing FP Added: 38 (22.35%) -> New Size: 208 | Added Acc: 0.9024
Iter  2 | Unadded Acc: 0.8293 | Missing FP Added: 55 (32.35%) -> New Size: 225 | Added Acc: 0.9024
Iter  3 | Unadded Acc: 0.8780 | Missing FP Added: 43 (25.29%) -> New Size: 213 | Added Acc: 0.9268
Iter  4 | Unadded Acc: 0.8780 | Missing FP Added: 51 (30.00%) -> New Size: 221 | Added Acc: 0.9024
Iter  5 | Unadded Acc: 0.8537 | Missing FP Added: 54 (31.76%) -> New Size: 224 | Added Acc: 0.9268
Iter  6 | Unadded Acc: 0.9024 | Missing FP Added: 48 (28.24%) -> New Size: 218 | Added Acc: 0.9268
Iter  7 | Unadded Acc: 0.9024 | Missing FP Added: 48 (28.24%) -> New Size: 218 | Added Acc: 0.9024
Iter  8 | Unadded Acc: 0.8293 | Missing FP Added: 50 (29.41%) -> New Size: 220 | Added Acc: 0.9024
Iter  9 | Unadded Acc: 0.9024 | Missing FP Added: 53 (31.18%) ->